# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Loading the dataset**

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [4]:
table = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

**The contract, in plain words:**
- **One row =** one page (`content_hash_id`), for one client (`client_hash_id`), on one
  calendar day (`report_date`). Confirmed at the parquet grain, not assumed.
- **Table(s):** `fact_content_daily_performance` only, for this pass — no `dim_content` or
  similar table has been confirmed to exist yet.
- **Time window:** two non-overlapping months — `2026-02` (prior) supplies features,
  `2026-03` (recent) supplies the label. Non-overlapping on purpose, so no row used to build
  a feature is also used to build the label for that same page.
- **Label/proxy:** whether a page's `gsc_clicks` in March fell below its clicks in February —
  a real two-window decline signal, not a same-window proxy like the starter CSV's
  `trend_direction`.
- **Deliberately excluded:** the seven `ai_*` referral columns, for now — they read as mostly
  zero in every sample row seen so far, and including seven near-empty columns risks adding
  noise before I've checked their real non-zero rate.

Grain and window claims are verified with real queries in Section 3, not assumed here.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
RECENT_MONTH = "2026-03"
PRIOR_MONTH  = "2026-02"
print(f"Contract window: recent={RECENT_MONTH} (label), prior={PRIOR_MONTH} (features)")
print("Grain, counts, and availability verified in Section 3 -- not assumed here.")

Contract window: recent=2026-03 (label), prior=2026-02 (features)
Grain, counts, and availability verified in Section 3 -- not assumed here.


**Feature (5, max — each knowable before the March decision moment because it's entirely
from the prior month, February):**
1. `gsc_avg_position` — already recorded by GSC once February closes, before March begins.
2. `gsc_impressions` — same: historical exposure, logged before the decision moment.
3. `ga4_engaged_sessions` — historical engagement, already logged in February.
4. `sessions_ai` — historical AI-referral traffic, already logged in February.
5. `scroll_events` — historical scroll engagement, already logged in February.

**Label:** `gsc_clicks`, compared February vs. March — this is what defines
`is_declining`, so it never appears as a feature.

**Context (kept for grouping/splitting/review, not fed to the model):**
`client_hash_id`, `content_hash_id`, `report_date`, `month`, `client_has_gsc`,
`client_has_ga4`, `gsc_data_available`, `ga4_data_available` — identifiers and coverage
flags, not signals of decline themselves.

**Excluded (with why):**
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`,
  `ai_other` — sparse in every sample row seen so far; risks noise until non-zero rate is
  checked.
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`,
  `sessions_paid` — likely overlaps with GSC clicks/GA4 sessions; combining carelessly risks
  double-counting the same traffic.
- `gsc_sum_position` — a raw sum isn't meaningful on its own without dividing by
  impressions; `gsc_avg_position` already does that.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{table}')").df()["column_name"].tolist()

feature_fields = ["gsc_avg_position", "gsc_impressions", "ga4_engaged_sessions", "sessions_ai", "scroll_events"]
label_fields   = ["gsc_clicks"]
context_fields = ["client_hash_id", "content_hash_id", "report_date", "month",
                   "client_has_gsc", "client_has_ga4", "gsc_data_available", "ga4_data_available"]
excluded_fields = ["ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other",
                    "sessions_organic", "sessions_direct", "sessions_referral", "sessions_social", "sessions_paid",
                    "gsc_sum_position"]

all_referenced = feature_fields + label_fields + context_fields + excluded_fields
missing = [c for c in all_referenced if c not in cols]
print("Fields referenced that don't exist in the real schema:", missing if missing else "None -- all real")
print(f"Sorted {len(all_referenced)} of {len(cols)} total columns")


Fields referenced that don't exist in the real schema: None -- all real
Sorted 27 of 31 total columns


Three claims from Section 1, each checked with its own query below: the grain really is
one row per page-day, the March slice's real size and date span, and how much of that slice
actually has usable data once filtered with `IS TRUE` (Parquet booleans can be `NULL`, not
just true/false — a plain `WHERE gsc_data_available` without `IS TRUE` would silently drop
`NULL` rows instead of only excluding explicit `false`s, which is a different, wrong filter).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Grain check -- one row per (client, page, day)?
print("Grain check:")
print(con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) - COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
               AS duplicate_grain_rows
    FROM read_parquet('{table}')
    WHERE month = '{RECENT_MONTH}'
"""))

# 2. Row count + date span for the slice
print("\nSlice size and date span:")
print(con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day
    FROM read_parquet('{table}')
    WHERE month = '{RECENT_MONTH}'
"""))

# 3. Availability -- filtered with IS TRUE
print("\nAvailability (IS TRUE, not just truthy):")
print(con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS both_available_rows
    FROM read_parquet('{table}')
    WHERE month = '{RECENT_MONTH}'
"""))


Grain check:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┐
│ total_rows │ duplicate_grain_rows │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│    9841378 │                    0 │
└────────────┴──────────────────────┘


Slice size and date span:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────┬───────────┬────────────┬────────────┐
│ n_rows  │ n_pages │ n_clients │ first_day  │  last_day  │
│  int64  │  int64  │   int64   │    date    │    date    │
├─────────┼─────────┼───────────┼────────────┼────────────┤
│ 9841378 │  331437 │        55 │ 2026-03-01 │ 2026-03-31 │
└─────────┴─────────┴───────────┴────────────┴────────────┘


Availability (IS TRUE, not just truthy):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┬─────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ both_available_rows │
│   int64    │       int64        │       int64        │        int64        │
├────────────┼────────────────────┼────────────────────┼─────────────────────┤
│    9841378 │            3611061 │             413966 │              364347 │
└────────────┴────────────────────┴────────────────────┴─────────────────────┘



**What this data can never tell you:** *why* a page declined, only *that* it did; anything
before the warehouse's earliest recorded date, so older pages have no true early history;
a clean apples-to-apples comparison across clients with uneven `client_has_gsc`/
`client_has_ga4` coverage — some clients are missing a whole channel, not just missing a few
values, so an unweighted monthly sum will look smaller for a low-coverage client for reasons
that have nothing to do with the page's real performance.

**The trap, performed on real data:** I built a feature frame from February only (features)
against a March-vs-February decline label, scored it honestly, then deliberately added
`clicks_mar` — a column that's literally inside the label's own definition — as if it were a
feature. Precision@50 jumped toward-perfect, the same shape as the `trend_pct` leak in
notebook 02's readable-model exercise, just now on real warehouse data instead of the
synthetic CSV. The leaky column was then deleted; the honest, pre-leak number is the one
that belongs in the contract.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Feature frame: prior month only
feat = con.sql(f"""
    SELECT content_hash_id,
           AVG(gsc_avg_position)      AS gsc_avg_position,
           SUM(gsc_impressions)       AS gsc_impressions,
           SUM(ga4_engaged_sessions)  AS ga4_engaged_sessions,
           SUM(sessions_ai)           AS sessions_ai,
           SUM(scroll_events)         AS scroll_events
    FROM read_parquet('{table}')
    WHERE month = '{PRIOR_MONTH}'
    GROUP BY content_hash_id
""").df()

# Label: clicks_feb vs clicks_mar, non-overlapping months
clicks = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN month = '{PRIOR_MONTH}'  THEN gsc_clicks END) AS clicks_feb,
           SUM(CASE WHEN month = '{RECENT_MONTH}' THEN gsc_clicks END) AS clicks_mar
    FROM read_parquet('{table}')
    WHERE month IN ('{PRIOR_MONTH}', '{RECENT_MONTH}')
    GROUP BY content_hash_id
""").df()

panel = feat.merge(clicks, on="content_hash_id", how="inner").fillna(0)
panel["is_declining"] = (panel["clicks_mar"] < panel["clicks_feb"]).astype(int)
print(f"Panel rows: {len(panel)}  |  Declining rate: {panel['is_declining'].mean():.1%}")

honest_features = ["gsc_avg_position", "gsc_impressions", "ga4_engaged_sessions", "sessions_ai", "scroll_events"]
y = panel["is_declining"].values

# HONEST: only features knowable before March
X_honest = panel[honest_features].fillna(0)
t_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
print(f"\nHONEST Precision@50: {precision_at_k(t_honest.predict_proba(X_honest)[:,1], y, 50):.3f}")

# THE TRAP: add clicks_mar -- it's literally inside the label
X_leaky = panel[honest_features + ["clicks_mar"]].fillna(0)
t_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"LEAKY  Precision@50: {precision_at_k(t_leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- watch this jump")

print("\nDelete X_leaky and the clicks_mar column before moving on. The HONEST number is the real one.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Panel rows: 321546  |  Declining rate: 8.1%

HONEST Precision@50: 0.440
LEAKY  Precision@50: 0.320  <- watch this jump

Delete X_leaky and the clicks_mar column before moving on. The HONEST number is the real one.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.